# Notebook 06：Genesis GPU環境の復旧＋クラッシュ診断（2026-10-16）

**診断ZIPの事実**：`torch==2.11.0+cpu`、CUDA不可、`genesis-world`未導入の段階で停止。以前のカーネル強制終了そのものの原因はなお未特定です。

**必須順序**：GPUが割り当てられたランタイムへ接続 → nvidia-smiが成功するまで待機 → CUDA対応torch確認/必要なら修復 → Genesis導入 → Stage 0（物理/描画）→ 学習済み資産 → 本編。

`HFH_READ`はNotebook 04で必要ですが、この06では使用しません。GPUが見えない場合にCPUへ切り替えて先へ進むことはしません。

## 0. 新規GPUランタイム
Colab「ランタイム」→「ランタイムのタイプを変更」→ **GPU**。以前のセッションとライブラリの混在を避けるため、再接続してから**上から順番に**実行してください。

最初のセルで `nvidia-smi` が失敗した場合、PyTorchやGenesisをインストールしてもGPUの割当は修復できません。ノートブックはそこで停止します。

In [14]:
# 調査結果: 以前の診断では torch==2.11.0+cpu, CUDA不可、Genesis未導入でした。
# ここではtorch/GenesisをNotebook本体にはimportせず、まずGPUハードウェアを確認します。
import sys
import os
import re
import json
import shutil
import platform
import subprocess
from pathlib import Path
OUT = Path('outputs')
OUT.mkdir(parents = True, exist_ok = True)
report = {
    'python': sys.version.split()[0],
    'executable': sys.executable,
    'nvidia_smi_path': shutil.which('nvidia-smi'),
    'nvidia_dev_present': Path('/dev/nvidia0').exists(),
}

def runinfo(cmd, timeout = 45):
    try:
        p = subprocess.run(cmd, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, timeout = timeout)
        return {'returncode':p.returncode, 'output':p.stdout[-12000:]}
    except Exception as ex:
        return {'returncode':None, 'error':type(ex).__name__ + ': ' + str(ex)}

report['nvidia_smi'] = runinfo(['nvidia-smi'])
report['torch_pip'] = runinfo([sys.executable,'-m','pip','show','torch'])
report['genesis_pip'] = runinfo([sys.executable,'-m','pip','show','genesis-world'])
(OUT/'environment_before.json').write_text(json.dumps(report, ensure_ascii = False, indent = 2), encoding = 'utf8')
print('Python:', report['python'], 'interpreter:', report['executable'])
print('nvidia-smi path:', report['nvidia_smi_path'])
print('GPU device /dev/nvidia0:', report['nvidia_dev_present'])
print('nvidia-smi:', report['nvidia_smi'].get('output', report['nvidia_smi'].get('error',''))[-2500:])
print('torch package:', report['torch_pip'].get('output','')[:650])
print('genesis package:', report['genesis_pip'].get('output','')[:550])
if report['nvidia_smi']['returncode'] != 0:
    raise RuntimeError(
        'GPUを確認できません。インストールに進まないでください。Colabの「ランタイム > ランタイムのタイプを変更」でGPUを選び、'
        '既存ランタイムを削除/再接続して、nvidia-smiを先に確認してください。\n'
        '診断ファイル: outputs/environment_before.json'
    )
raw = report['nvidia_smi'].get('output','')
m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', raw)
report['driver_cuda_max'] = m.group(0) if m else None
(OUT/'environment_before.json').write_text(json.dumps(report, ensure_ascii = False, indent = 2), encoding = 'utf8')
if m and (int(m.group(1)), int(m.group(2))) < (12, 6):
    raise RuntimeError('表示されたドライバの対応CUDAが12.6未満です。このNotebookのPyTorch 2.11/cu126構成とは整合しません。別のGPUランタイム/互換版を検討してください。')
print('GPUが確認されました。次にPyTorchとGenesisを検証・導入します。')

Python: 3.13.15 interpreter: /usr/bin/python3
nvidia-smi path: /opt/bin/nvidia-smi
GPU device /dev/nvidia0: True
nvidia-smi: Fri Oct 16 05:54:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       3MiB /  97887MiB |      0%      Default |
|              

## 1. PyTorch CUDA版の確認・修復とGenesis導入
**Genesis公式は、PyTorchを先にインストールしてからGenesisを導入する手順を推奨しています。**

PyTorchが既にCUDA対応なら再インストールは行いません。CPU専用ビルドの場合のみ、PyTorch公式CUDA 12.6 wheelを再導入します。ドライバの対応CUDAが12.6未満なら手動で互換性を調べてください。

**注意**：大容量ダウンロードが発生する場合があります。成功後は別プロセスでバージョンとGPU利用可否を再検証します。Python環境を変更した後でimportエラーが続く場合はランタイムを再起動して再実行してください（GPU確認から）。

In [15]:
# torchはsubprocessだけでimportし、Notebook本体のメモリ上に古いCPU版を残しません。
import sys
import os
import subprocess
import json
import time
from pathlib import Path
out = Path('outputs')
out.mkdir(parents = True, exist_ok = True)

def command_logged(cmd, filename, timeout = 1800):
    log = out/filename
    print('RUN:', ' '.join(cmd[:5]), '...（出力は', log, 'へ保存）', flush = True)
    start = time.monotonic()
    with log.open('w', encoding = 'utf-8') as f:
        try:
            p = subprocess.run(cmd, stdout = f, stderr = subprocess.STDOUT, text = True, timeout = timeout)
            rc = p.returncode
        except subprocess.TimeoutExpired:
            rc = 124
    print(filename, 'exit:', rc,'elapsed_s:', round(time.monotonic()-start, 1))
    if rc:
        print('\n'.join(log.read_text(errors = 'replace').splitlines()[-28:]))
        raise RuntimeError(f'{filename}が失敗しました（終了コード{rc}）。この段階で停止しログを確認してください。')

verify_torch = [sys.executable,'-c',
    'import torch; print("torch",torch.__version__,"torch.version.cuda",torch.version.cuda,"available",torch.cuda.is_available());'
    'assert torch.cuda.is_available(), "CUDA enabled torch wheel/device is required";'
    'print("device",torch.cuda.get_device_name(0))']

check = subprocess.run(verify_torch, capture_output = True, text = True, timeout = 120)
print('CUDA検証（導入前）:', check.stdout, check.stderr[-1000:])
if check.returncode != 0:
    # PyTorch公式 v2.11.0: CUDA 12.6用インデックス。CUDA GPUが実際に割り当てられた場合にのみ実行。
    command_logged([
        sys.executable,'-m','pip','install','--upgrade','--force-reinstall',
        'torch==2.11.0','torchvision==0.26.0',
        '--index-url','https://download.pytorch.org/whl/cu126'
    ], 'pip_torch_cuda.log', timeout = 2400)
# CUDAを修復できたことを**先に**確認。失敗したらGenesisをインストールしない。
command_logged(verify_torch,'verify_cuda_torch.log', timeout = 180)

# Ubuntu/Colab のオフスクリーンレンダリング関連共有ライブラリ。実行不可ならログを確認。
if Path('/usr/bin/apt-get').exists() and os.geteuid() == 0:
    command_logged(['apt-get','update','-qq'],'apt_update.log', timeout = 540)
    command_logged(['apt-get','install','-y','-qq','libegl1','libgl1','libgl1-mesa-dri','libosmesa6','ffmpeg'],
                   'apt_graphics.log', timeout = 540)

command_logged([sys.executable,'-m','pip','install',
     'genesis-world==1.4.4','numpy<3','scipy<2','scikit-learn>=1.3,<2',
     'pandas>=2,<3','opencv-python-headless>=4.8,<5','pillow','psutil','imageio-ffmpeg'],
     'pip_genesis.log', timeout = 1800)

# Genesis導入がtorchをCPU版に戻していないか再確認する。
command_logged(verify_torch,'verify_cuda_after_genesis.log', timeout = 180)
command_logged([sys.executable,'-c',
   'import importlib.metadata as md; import genesis as gs; print("genesis-world",md.version("genesis-world"))'],
   'verify_genesis_import.log', timeout = 180)
print('環境の準備が完了しました。Stage 0A → 0B → 0C に進んでください。')

CUDA検証（導入前）: torch 2.11.0+cu130 torch.version.cuda 13.0 available True
device NVIDIA RTX PRO 6000 Blackwell Server Edition
 
RUN: /usr/bin/python3 -c import torch; print("torch",torch.__version__,"torch.version.cuda",torch.version.cuda,"available",torch.cuda.is_available());assert torch.cuda.is_available(), "CUDA enabled torch wheel/device is required";print("device",torch.cuda.get_device_name(0)) ...（出力は outputs/verify_cuda_torch.log へ保存）
verify_cuda_torch.log exit: 0 elapsed_s: 0.9
RUN: apt-get update -qq ...（出力は outputs/apt_update.log へ保存）
apt_update.log exit: 0 elapsed_s: 1.5
RUN: apt-get install -y -qq libegl1 ...（出力は outputs/apt_graphics.log へ保存）
apt_graphics.log exit: 0 elapsed_s: 1.0
RUN: /usr/bin/python3 -m pip install genesis-world==1.4.4 ...（出力は outputs/pip_genesis.log へ保存）
pip_genesis.log exit: 0 elapsed_s: 1.3
RUN: /usr/bin/python3 -c import torch; print("torch",torch.__version__,"torch.version.cuda",torch.version.cuda,"available",torch.cuda.is_available());assert torch.cu

## Stage 0：最小構成のGenesis起動・物理・描画（HF学習済みZIP不要）

ここで失敗したら、AIや巡回アルゴリズムは原因ではありません。全て別プロセスで実行します。**上から一つずつ実行し、失敗したら次に進まないでください。**

- 0A：CUDA上でGenesisを初期化（`P0_init.log`）
- 0B：床の3ステップだけを物理計算（`P0_physics.log`）
- 0C：小さな画像を1枚だけ描画（`P0_render.log`）

0Bはカメラなしですが、GenesisのScene構築では内部描画ライブラリに依存することがあります。0B成功・0C失敗なら、EGL/OpenGLを優先して調べます。

**今回の診断はtorch段階で止まっており、GenesisのInit/物理/描画は未検査です。環境準備の成功はGenesisの3D録画成功を意味しません。**

In [16]:
%%writefile genesis_minimal_probe.py
"""Genesis headless smoke test. Keeps native crashes outside the Jupyter kernel.
Test only: GPU physics + optional single offscreen rendered image.
"""
import os
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
os.environ.setdefault('PYGLET_HEADLESS', 'true')
import argparse
import json
import time
import traceback
from pathlib import Path

def marker(stage, **details):
    print(json.dumps({'stage':stage,'ts':time.time(), **details}, ensure_ascii = False, default = str), flush = True)

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices = ['init','physics','render'], required = True)
    p.add_argument('--output', default = 'outputs')
    args = p.parse_args()
    out = Path(args.output)
    out.mkdir(parents = True, exist_ok = True)
    marker('process_start', mode = args.mode, gl = os.environ.get('PYOPENGL_PLATFORM'))
    import torch
    marker('torch_check', torch_version = torch.__version__, torch_cuda = bool(torch.cuda.is_available()))
    if not torch.cuda.is_available():
        raise RuntimeError('PyTorch CUDA is unavailable; stop here. TensorFlow warning alone is not decisive.')
    import genesis as gs
    marker('genesis_imported', version = getattr(gs,'__version__','unknown'))
    gs.init(backend = gs.cuda, logging_level = 'warning', seed = 42)
    marker('genesis_initialized', backend = str(gs.backend), device = str(gs.device))
    if args.mode == 'init':
        return
    marker('scene_setup_begin')
    scene = gs.Scene(show_viewer = False, sim_options = gs.options.SimOptions(dt = 0.05))
    scene.add_entity(gs.morphs.Plane())
    camera = None
    if args.mode == 'render':
        camera = scene.add_camera(res = (160, 120), pos = (2.3, -2.8, 2.0), lookat = (0, 0, 0), GUI = False)
    marker('scene_build_begin')
    scene.build()
    marker('scene_build_success')
    for i in range(3):
        scene.step()
        marker('scene_step_success', step = i+1)
    if camera is not None:
        marker('render_begin')
        rgb, _, _, _ = camera.render()
        assert rgb is not None
        import numpy as np
        from PIL import Image
        img = np.asarray(rgb)
        Image.fromarray(img.astype('uint8')).save(out/'minimal_genesis_render.png')
        marker('render_success', shape = list(img.shape))
    marker('probe_success', mode = args.mode)

if __name__ == '__main__':
    try:
        main()
    except BaseException as exc:
        marker('probe_error', type = type(exc).__name__, reason = str(exc))
        traceback.print_exc()
        raise


Overwriting genesis_minimal_probe.py


In [17]:
import subprocess
import sys
import os
import time
import json
from pathlib import Path
probe_output = Path('outputs')
probe_output.mkdir(exist_ok = True, parents = True)

def run_probe(mode, timeout_s = 600):
    logfile=probe_output/f'P0_{mode}.log'
    env = os.environ.copy()
    env['PYOPENGL_PLATFORM'] = os.environ.get('PAI_GL','egl')
    env['PYGLET_HEADLESS'] = 'true'
    command = [sys.executable,'-u','genesis_minimal_probe.py','--mode', mode,'--output', str(probe_output)]
    start = time.monotonic()
    with logfile.open('w', encoding = 'utf8') as f:
        try:
            completed=subprocess.run(command,stdout=f,stderr=subprocess.STDOUT,env=env,
                                     timeout = timeout_s, check = False)
            result = completed.returncode
        except subprocess.TimeoutExpired:
            result = 'timeout'
    tail = '\n'.join(logfile.read_text(errors = 'replace').splitlines()[-20:])
    print(f'Probe {mode}: exit={result}, wall={time.monotonic()-start:.1f}s, log={logfile}')
    print(tail)
    if result != 0:
        raise RuntimeError(f'Genesis minimal probe {mode} failed (exit={result}); '
                           f'download {logfile} and STOP before running mission')

print('Run the following three cells IN ORDER; do not use Run all if any fails.')

Run the following three cells IN ORDER; do not use Run all if any fails.


In [18]:
run_probe("init", timeout_s = 420)

Probe init: exit=0, wall=5.5s, log=outputs/P0_init.log
{"stage": "process_start", "ts": 1792130091.1321187, "mode": "init", "gl": "egl"}
{"stage": "torch_check", "ts": 1792130091.7869575, "torch_version": "2.11.0+cu130", "torch_cuda": true}
{"stage": "genesis_imported", "ts": 1792130095.4731917, "version": "1.4.4"}
{"stage": "genesis_initialized", "ts": 1792130095.6113853, "backend": "2", "device": "cuda:0"}


In [19]:
run_probe("physics", timeout_s = 600)

Probe physics: exit=0, wall=6.7s, log=outputs/P0_physics.log
{"stage": "process_start", "ts": 1792130096.6617727, "mode": "physics", "gl": "egl"}
{"stage": "torch_check", "ts": 1792130097.3300712, "torch_version": "2.11.0+cu130", "torch_cuda": true}
{"stage": "genesis_imported", "ts": 1792130101.050512, "version": "1.4.4"}
{"stage": "genesis_initialized", "ts": 1792130101.191353, "backend": "2", "device": "cuda:0"}
{"stage": "scene_setup_begin", "ts": 1792130101.1914}
{"stage": "scene_build_begin", "ts": 1792130101.434209}
{"stage": "scene_build_success", "ts": 1792130102.1439803}
{"stage": "scene_step_success", "ts": 1792130102.2066908, "step": 1}
{"stage": "scene_step_success", "ts": 1792130102.2071772, "step": 2}
{"stage": "scene_step_success", "ts": 1792130102.2075005, "step": 3}
{"stage": "probe_success", "ts": 1792130102.2075121, "mode": "physics"}


In [20]:
run_probe("render", timeout_s = 600)

Probe render: exit=0, wall=9.1s, log=outputs/P0_render.log
{"stage": "process_start", "ts": 1792130103.390754, "mode": "render", "gl": "egl"}
{"stage": "torch_check", "ts": 1792130104.0401855, "torch_version": "2.11.0+cu130", "torch_cuda": true}
{"stage": "genesis_imported", "ts": 1792130107.7130342, "version": "1.4.4"}
{"stage": "genesis_initialized", "ts": 1792130107.8498335, "backend": "2", "device": "cuda:0"}
{"stage": "scene_setup_begin", "ts": 1792130107.8498795}
{"stage": "scene_build_begin", "ts": 1792130108.0925817}
{"stage": "scene_build_success", "ts": 1792130109.809808}
{"stage": "scene_step_success", "ts": 1792130109.87245, "step": 1}
{"stage": "scene_step_success", "ts": 1792130109.8728652, "step": 2}
{"stage": "scene_step_success", "ts": 1792130109.8731914, "step": 3}
{"stage": "render_begin", "ts": 1792130109.8732014}
{"stage": "render_success", "ts": 1792130109.9604716, "shape": [120, 160, 3]}
{"stage": "probe_success", "ts": 1792130109.9605048, "mode": "render"}


## 2. 以前の評価用コード／URDF生成コードを配置
Notebook 06から関連コードを引き継ぎ、**無録画物理テストではカメラを作らない**ようにしています。

In [21]:
%%writefile pai_patrol.py
"""Physical AI: risk-aware mobile inspection robot (Colab, reproducible demo).

Simulation fidelity: 2-D occupancy grid + differential-drive/unicycle kinematics,
not a ROS/Gazebo or hardware experiment. Remote sensors are synthetic, not measured.
Anomaly ground truth is private to the environment and used only for scoring.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import heapq
import math
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

STATIONS = {
    'A': (1.5, 7.4),
    'B': (3.5, 6.1),
    'C': (8.6, 7.0),
    'D': (10.5, 3.4),
    'E': (6.5, 1.3),
    'F': (2.0, 3.6),
}
BASE = (1.0, 1.0)
OBSTACLES = [(3.9, 2.1, 5.0, 4.8), (6.5, 4.1, 8.0, 5.7), (8.1, 1.5, 9.0, 3.2)]
WIDTH, HEIGHT = 12.0, 9.0
FEATURES = ['temperature_C', 'vibration_mm_s', 'current_A', 'sound_dB']
STATION_KEYS = list(STATIONS)
MEANS = np.array([[35 + i*1.2, 2.1+i*.08, 5+i*.13, 51+i*.45] for i in range(len(STATIONS))], dtype = float)
STDS = np.array([1.6, .35, .26, 2.1])
FAULT_SHIFT = np.array([9.0, 1.8, 1.2, 9.0])

@dataclass(frozen = True)
class Config:
    dt: float = 0.10
    horizon: float = 150.0
    max_speed: float = 0.68
    max_turn: float = 2.4
    robot_radius: float = 0.25
    grid_res: float = 0.25
    remote_period: float = 2.0
    camera_fps: int = 10
    camera_stride: int = 1
    risk_gate: float = 1.10
    revisit_after: float = 15.0

CFG = Config()

class FloorMap:
    def __init__(self, cfg = CFG):
        self.cfg = cfg
        self.nx = int(round(WIDTH/cfg.grid_res))
        self.ny = int(round(HEIGHT/cfg.grid_res))
        self.grid = np.zeros((self.ny, self.nx), dtype = np.uint8)
        for gy in range(self.ny):
            for gx in range(self.nx):
                x, y = self.from_cell((gx, gy))
                if self.blocked(x, y, margin = cfg.robot_radius+0.09):
                    self.grid[gy, gx] = 1
        assert all(self.grid[self.to_cell(pt)[1], self.to_cell(pt)[0]] == 0 for pt in [BASE, *STATIONS.values()]), 'Station collides with inflated obstacles'
    def blocked(self, x, y, margin = 0.):
        if x<margin or y<margin or x>WIDTH-margin or y>HEIGHT-margin:
            return True
        return any(a-margin <= x <= c+margin and b-margin <= y <= d+margin for a, b, c, d in OBSTACLES)
    def to_cell(self, xy):
        return (min(self.nx-1,max(0,int(xy[0]/self.cfg.grid_res))),
                min(self.ny-1, max(0, int(xy[1]/self.cfg.grid_res))))
    def from_cell(self, cell):
        return ((cell[0]+.5)*self.cfg.grid_res, (cell[1]+.5)*self.cfg.grid_res)
    def line_clear(self, a, b):
        d = math.dist(a, b)
        for u in np.linspace(0, 1, max(2, int(math.ceil(d/0.09)))):
            x = a[0]+(b[0]-a[0])*u
            y = a[1]+(b[1]-a[1])*u
            if self.blocked(x, y, self.cfg.robot_radius+0.04):
                return False
        return True
    def plan(self, start, end):
        s = self.to_cell(start)
        g = self.to_cell(end)
        if s == g or self.line_clear(start, end):
            return [tuple(end)]
        h = lambda p: math.hypot(p[0]-g[0], p[1]-g[1])
        heap = [(h(s), 0., s)]
        best = {s:0.}
        parent = {}
        found = False
        dirs = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if dx or dy]
        while heap:
            _, cost, u = heapq.heappop(heap)
            if cost>best.get(u, float('inf'))+1e-8:
                continue
            if u == g:
                found = True
                break
            for dx, dy in dirs:
                v = (u[0]+dx, u[1]+dy)
                if not (0 <= v[0]<self.nx and 0 <= v[1]<self.ny) or self.grid[v[1], v[0]]:
                    continue
                if dx and dy and (self.grid[u[1], v[0]] or self.grid[v[1], u[0]]):
                    continue
                nc = cost+math.hypot(dx, dy)
                if nc<best.get(v, float('inf')):
                    best[v] = nc
                    parent[v] = u
                    heapq.heappush(heap, (nc+h(v), nc, v))
        if not found:
            raise RuntimeError(f'Route unreachable: {start} -> {end}')
        path = [g]
        while path[-1] != s:
            path.append(parent[path[-1]])
        path.reverse()
        pts = [tuple(start)]+[self.from_cell(pt) for pt in path[1:-1]]+[tuple(end)]
        # Greedy line-of-sight waypoint reduction (collision checked, inflated obstacles)
        out = []
        i = 0
        while i<len(pts)-1:
            j = len(pts)-1
            while j>i+1 and not self.line_clear(pts[i], pts[j]):
                j -= 1
            out.append(pts[j])
            i = j
        return out
    def distance(self, start, end):
        p = [tuple(start)]+self.plan(start, end)
        return sum(math.dist(p[i], p[i+1]) for i in range(len(p)-1))

@dataclass(frozen = True)
class Fault:
    station: str
    onset: float
    severity: float

@dataclass(frozen = True)
class Scenario:
    seed: int
    faults: tuple[Fault, ...]


def make_scenario(seed):
    rng = np.random.default_rng(int(seed))
    keys = rng.choice(STATION_KEYS, size = 2, replace = False)
    return Scenario(int(seed),(Fault(str(keys[0]),0.0,float(rng.uniform(.95,1.45))),
                               Fault(str(keys[1]), float(rng.choice([12., 18., 24.])), float(rng.uniform(1.05, 1.55)))))


def telemetry(scenario, station, t, mode = 'remote', replicate = 0):
    """Synthetic sensor. Policy receives sensor values only, never `scenario.faults`."""
    idx = STATION_KEYS.index(station)
    # Deterministic per (seed, equipment, time-window, mode, replicate).
    tick = max(0, int(t//(CFG.remote_period if mode == 'remote' else 1.0)))
    ss = np.random.SeedSequence([scenario.seed, idx, tick, 0 if mode == 'remote' else 1, int(replicate)])
    rng = np.random.default_rng(ss)
    noise = 1.17 if mode == 'remote' else .40
    x = MEANS[idx]+rng.normal(size = 4)*STDS*noise
    for fault in scenario.faults:
        if station == fault.station and t >= fault.onset:
            x = x+FAULT_SHIFT*fault.severity
    return x

class Detector:
    def __init__(self, seed = 2026):
        rng = np.random.default_rng(seed)
        normal = rng.normal(size = (3500, 4))*1.17
        # Only NORMAL observations in training. Ground truth of evaluation faults unseen.
        self.model = IsolationForest(n_estimators = 120, max_samples = 256, random_state = seed, n_jobs = -1)
        self.model.fit(normal)
        vr = rng.normal(size = (3000, 4))*1.17
        vl = rng.normal(size = (3000, 4))*.40
        rs = -self.model.score_samples(vr)
        ls = -self.model.score_samples(vl)
        self.remote_threshold = float(np.quantile(rs, .99))
        self.local_threshold = float(np.quantile(ls, .995))
        self.reference_75 = float(np.quantile(rs, .75))
        self.spread = max(.015, self.remote_threshold-self.reference_75)
    def features(self, x, station):
        idx = STATION_KEYS.index(station)
        return ((np.asarray(x)-MEANS[idx])/STDS).reshape(1, -1)
    def anomaly(self, x, station):
        return float(-self.model.score_samples(self.features(x, station))[0])
    def remote_risk(self, x, station):
        a = self.anomaly(x, station)
        return float(np.clip((a-self.reference_75)/self.spread, 0., 3.0))
    def onsite_flag(self, scenario, station, t):
        # independent repeat-measurement median reduces occasional false alarms
        scores = [self.anomaly(telemetry(scenario, station, t, mode = 'local', replicate = k), station) for k in range(3)]
        med = float(np.median(scores))
        return med>self.local_threshold, med

@dataclass
class PatrolResult:
    policy: str
    scenario: Scenario
    metrics: dict
    visits: list
    trajectory: list
    events: list
    remote_history: list

class Patrol:
    def __init__(self, scenario, detector, policy = 'adaptive', cfg = CFG, floor = None):
        if policy not in ('adaptive','nearest'):
            raise ValueError('policy must be adaptive or nearest')
        self.scenario = scenario
        self.detector = detector
        self.policy = policy
        self.cfg = cfg
        self.floor = floor if floor is not None else FloorMap(cfg)
        self.pos = np.array(BASE, dtype = float)
        self.heading = math.pi/2
        self.t = 0.
        self.distance = 0.
        self.collisions = 0
        self.replans = 0
        self.route = []
        self.target = None
        self.waypoint = 0
        self.visits = []
        self.events = []
        self.history = []
        self.trajectory = []
        self.last_visit = {k:-1e9 for k in STATION_KEYS}
        self.visit_counts = {k:0 for k in STATION_KEYS}
        self.target_start = -1e9
        self.consecutive_alarm = {k:0 for k in STATION_KEYS}
        self.remote_risks = {k:0. for k in STATION_KEYS}
        self.active_alarm = {k:False for k in STATION_KEYS}
        self.detections = {}
        self.flagged_stations = set()
        self.fp_count = 0
        self.next_remote = 0.
        self.finished = False
    def observe_remote(self):
        risks = {}
        for key in STATION_KEYS:
            observed = telemetry(self.scenario, key, self.t,'remote')
            risk = self.detector.remote_risk(observed, key)
            self.remote_risks[key] = .35*self.remote_risks[key]+.65*risk
            self.consecutive_alarm[key] = (self.consecutive_alarm[key]+1 if self.remote_risks[key] >= self.cfg.risk_gate else 0)
            risks[key] = self.remote_risks[key]
        self.history.append({'t':round(self.t, 1), **risks})
    def options(self):
        outstanding = [k for k in STATION_KEYS if self.last_visit[k]<0]
        if self.policy == 'nearest':
            return outstanding
        alerts=[k for k in STATION_KEYS if self.consecutive_alarm[k]>=2
                and k not in self.flagged_stations and self.visit_counts[k]<2
                and (self.t-self.last_visit[k])>self.cfg.revisit_after]
        return sorted(set(outstanding+alerts))
    def choose_target(self):
        choices = self.options()
        if not choices:
            return 'HOME'
        distances = {k:self.floor.distance(self.pos, STATIONS[k]) for k in choices}
        if self.policy == 'nearest':
            return min(choices, key = lambda k:(distances[k], k))
        # Alarm priority competes with travel cost; no hidden ground truth labels used.
        return max(choices,key=lambda k:(3.5*self.remote_risks[k]-.30*distances[k]
                                         + (1.8 if self.last_visit[k]<0 else 0), -distances[k]))
    def set_target(self, k):
        if self.target == k:
            return
        self.target = k
        self.replans += 1
        self.target_start = self.t
        end = BASE if k == 'HOME' else STATIONS[k]
        self.route = self.floor.plan(self.pos, end)
        self.waypoint = 0
        self.events.append({'t':round(self.t, 1),'kind':'reroute','station':k})
    def arrive(self):
        k = self.target
        if k == 'HOME':
            self.finished = True
            self.events.append({'t':round(self.t, 1),'kind':'home','station':'HOME'})
            return
        flagged, score = self.detector.onsite_flag(self.scenario, k, self.t)
        fault = next((f for f in self.scenario.faults if f.station == k and self.t >= f.onset), None)
        is_fault = fault is not None
        if flagged:
            self.flagged_stations.add(k)
        if flagged and is_fault:
            self.detections.setdefault(k, self.t)
        if flagged and not is_fault:
            self.fp_count += 1
        self.last_visit[k] = self.t
        self.visit_counts[k] += 1
        row={'t':round(self.t,1),'station':k,'flagged':bool(flagged),'fault_active':bool(is_fault),
             'anomaly_score':score,'risk':self.remote_risks[k]}
        self.visits.append(row)
        self.events.append({'t':round(self.t,1),'kind':'inspect','station':k,
                            'flagged':bool(flagged),'fault_active':bool(is_fault)})
        self.target = None
        self.route = []
    def step(self):
        if self.finished:
            return
        if self.t+1e-8 >= self.next_remote:
            self.observe_remote()
            self.next_remote += self.cfg.remote_period
            if self.policy == 'adaptive' and self.target not in (None,'HOME') and self.t-self.target_start>6.:
                # A fresh strong remote alarm can interrupt a low-risk current route.
                opts = self.options()
                serious = [k for k in opts if k != self.target and self.consecutive_alarm[k] >= 2 and self.remote_risks[k]>self.remote_risks.get(self.target, 0)+1.05]
                if serious:
                    best = max(serious, key = lambda k:self.remote_risks[k]-.18*self.floor.distance(self.pos, STATIONS[k]))
                    self.set_target(best)
        if self.target is None:
            self.set_target(self.choose_target())
        goal = np.asarray(BASE if self.target == 'HOME' else STATIONS[self.target])
        if math.dist(self.pos, goal)<.37:
            self.arrive()
        else:
            if self.waypoint >= len(self.route):
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
            wp = np.asarray(self.route[self.waypoint])
            if np.linalg.norm(wp-self.pos)<.22 and self.waypoint<len(self.route)-1:
                self.waypoint += 1
                wp = np.asarray(self.route[self.waypoint])
            desired = math.atan2(wp[1]-self.pos[1], wp[0]-self.pos[0])
            delta = (desired-self.heading+math.pi)%(2*math.pi)-math.pi
            omega = np.clip(3.0*delta, -self.cfg.max_turn, self.cfg.max_turn)
            # Differential drive kinematics: v=(r/2)(wr+wl), w=(r/L)(wr-wl)
            v = self.cfg.max_speed * max(.05, 1.0-min(1.0, abs(delta)/1.3))
            # Convert commanded twist to actual left/right wheel rates, then integrate.
            wheel_radius, wheel_separation = .085, .36
            wheel_l = (v-.5*wheel_separation*omega)/wheel_radius
            wheel_r = (v+.5*wheel_separation*omega)/wheel_radius
            v_eff = .5*wheel_radius*(wheel_r+wheel_l)
            omega_eff = wheel_radius*(wheel_r-wheel_l)/wheel_separation
            next_heading = (self.heading+omega_eff*self.cfg.dt+math.pi)%(2*math.pi)-math.pi
            proposed = self.pos+v_eff*self.cfg.dt*np.array([math.cos(next_heading), math.sin(next_heading)])
            if self.floor.blocked(*proposed, margin = self.cfg.robot_radius+.01):
                self.collisions += 1
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
                v = 0.0
            else:
                self.pos = proposed
                self.distance += v*self.cfg.dt
            self.heading = next_heading
        self.t += self.cfg.dt
        self.trajectory.append((self.t,float(self.pos[0]),float(self.pos[1]),float(self.heading),self.target or '-',
                                float(max(self.remote_risks.values())), len(self.detections)))
    def run(self):
        nsteps = int(self.cfg.horizon/self.cfg.dt)
        for _ in range(nsteps):
            if self.finished:
                break
            self.step()
        fault_count = len(self.scenario.faults)
        delays = []
        for f in self.scenario.faults:
            found = self.detections.get(f.station)
            # Missed events receive the horizon penalty (censored observation).
            delays.append((self.cfg.horizon-f.onset if found is None else max(0, found-f.onset), f.severity))
        weight = sum(sev for _, sev in delays)
        weighted = sum(d*sev for d, sev in delays)/weight if weight else float('nan')
        inspected = len([k for k in STATION_KEYS if self.last_visit[k] >= 0])
        metrics={'seed':self.scenario.seed,'policy':self.policy,'detected':len(self.detections),
                 'faults':fault_count,'recall':len(self.detections)/fault_count,
                 'false_alarm_inspections':self.fp_count,
                 'weighted_detection_delay_s':weighted,'distance_m':self.distance,
                 'elapsed_s':self.t,'coverage':inspected/len(STATION_KEYS),
                 'replans':self.replans,'collision_attempts':self.collisions,
                 'home_reached':self.finished}
        return PatrolResult(self.policy, self.scenario, metrics, self.visits, self.trajectory, self.events, self.history)


def evaluate(seed_list = range(30, 50), detector = None, cfg = CFG, verbose = True):
    detector = detector if detector is not None else Detector()
    floor = FloorMap(cfg)
    results = []
    scenarios = []
    for seed in seed_list:
        s = make_scenario(seed)
        scenarios.append(s)
        for policy in ('nearest','adaptive'):
            results.append(Patrol(s, detector, policy, cfg, floor).run())
    dataframe = pd.DataFrame([r.metrics for r in results])
    paired = dataframe.pivot(index = 'seed', columns = 'policy')
    differences=pd.DataFrame({
        'delay_improvement_s':paired[('weighted_detection_delay_s','nearest')]-paired[('weighted_detection_delay_s','adaptive')],
        'recall_improvement':paired[('recall','adaptive')]-paired[('recall','nearest')],
        'extra_distance_m':paired[('distance_m','adaptive')]-paired[('distance_m','nearest')],
    })
    if verbose:
        print('Policies: nearest = distance-only patrol; adaptive = AI-risk priority patrol')
        print('Note: training/calibration and evaluation scenarios are separate.')
        print(dataframe.groupby('policy')[['recall','weighted_detection_delay_s','distance_m','coverage','collision_attempts','false_alarm_inspections']].mean().round(2).to_string())
        print('Paired mean differences (positive delay_improvement is better):')
        print(differences.mean().round(2).to_string())
    return dataframe, differences, results


def smoke_test():
    floor = FloorMap()
    d = Detector()
    for k, p in STATIONS.items():
        assert floor.plan(BASE, p), k
    scenario = make_scenario(42)
    v = Patrol(scenario, d,'adaptive', floor = floor).run()
    assert 0 <= v.metrics['recall'] <= 1
    assert len(v.trajectory)>40
    assert not np.isnan(v.metrics['weighted_detection_delay_s'])
    assert v.metrics['elapsed_s'] <= CFG.horizon+1e-6
    print('smoke test OK:', {k:round(v.metrics[k], 2) if isinstance(v.metrics[k], float) else v.metrics[k] for k in ['recall','distance_m','elapsed_s','coverage','home_reached','collision_attempts']})
    return v

if __name__ == '__main__':
    smoke_test()
    evaluate(range(42, 46))


Overwriting pai_patrol.py


In [22]:
%%writefile genesis_mission.py
"""Genesis World 3-D rigid-body mobile rover driven by wheel velocity commands.

All observations are synthetic; the simulator is a *simulated* factory. No hardware validation.
The policy receives only synthetic sensor observations, never ground-truth fault IDs/times.
3-D dynamic motion is separately logged; planner-only benchmark is a DIFFERENT experiment.
"""
from __future__ import annotations
import json
import math
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from pai_patrol import (BASE, STATIONS, OBSTACLES, STATION_KEYS, CFG,
                        Detector, Patrol, FloorMap, make_scenario, Scenario)

@dataclass
class MissionSettings:
    dt: float = 0.05
    duration_s: float = 90.0
    fps: int = 10
    backend: str = 'cuda'  # 'cpu' option for diagnostics
    res: tuple[int, int] = (640, 480)
    policy: str = 'adaptive'  # 'nearest' or 'adaptive'
    seed: int = 42
    max_v: float = 0.65
    max_omega: float = 1.9
    wheel_radius: float = 0.12
    wheel_track: float = 0.41
    # URDF positive y wheel axes rotate the rover toward -x for positive omega_y
    wheel_sign: float = -1.0


def generate_rover_urdf(path: Path):
    """Create a simple explicitly inertialized differential drive URDF, no external assets."""
    path = Path(path)
    path.parent.mkdir(parents = True, exist_ok = True)
    text = '''<?xml version="1.0"?>
<robot name="pai_differential_rover">
 <material name="blue"><color rgba="0.13 0.39 0.78 1"/></material>
 <material name="dark"><color rgba="0.12 0.15 0.20 1"/></material>
 <material name="accent"><color rgba="0.95 0.65 0.20 1"/></material>
 <link name="base_link">
   <inertial><origin xyz="0 0 0"/><mass value="6"/><inertia ixx="0.14" ixy="0" ixz="0" iyy="0.18" iyz="0" izz="0.20"/></inertial>
   <visual><geometry><box size="0.50 0.32 0.13"/></geometry><material name="blue"/></visual>
   <collision><geometry><box size="0.50 0.32 0.13"/></geometry></collision>
 </link>
 <link name="left_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="left_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="left_wheel"/><origin xyz="-0.05 0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="right_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="right_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="right_wheel"/><origin xyz="-0.05 -0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="caster_ball">
  <inertial><origin xyz="0 0 0"/><mass value="0.1"/><inertia ixx="0.00015" ixy="0" ixz="0" iyy="0.00015" iyz="0" izz="0.00015"/></inertial>
  <visual><geometry><sphere radius="0.06"/></geometry><material name="accent"/></visual>
  <collision><geometry><sphere radius="0.06"/></geometry></collision>
 </link>
 <joint name="caster_joint" type="fixed"><parent link="base_link"/><child link="caster_ball"/><origin xyz="0.19 0 -0.19"/></joint>
</robot>'''
    path.write_text(text, encoding = 'utf8')
    return path


def as_numpy(t):
    return t.detach().cpu().numpy() if hasattr(t, 'detach') else np.asarray(t)


def xy_yaw(robot):
    xyz = as_numpy(robot.get_pos()).reshape(-1, 3)[0]
    w, x, y, z = as_numpy(robot.get_quat()).reshape(-1, 4)[0]
    yaw = math.atan2(2*(w*z+x*y), 1-2*(y*y+z*z))
    return (float(xyz[0]), float(xyz[1])), float(yaw), float(xyz[2])


def make_scene(settings:MissionSettings, output:Path, enable_camera = True):
    import genesis as gs
    if settings.dt <= 0 or settings.duration_s <= 0:
        raise ValueError('Invalid time settings')
    output = Path(output)
    output.mkdir(parents = True, exist_ok = True)
    scene=gs.Scene(show_viewer=False,
                   sim_options = gs.options.SimOptions(dt = settings.dt),
                   vis_options = gs.options.VisOptions(show_world_frame = False))
    scene.add_entity(gs.morphs.Plane(), surface = gs.surfaces.Plastic(color = (.80, .84, .88, 1)))
    # Ground obstacles: rectangles come from same grid as the planner.
    for i, (x1, y1, x2, y2) in enumerate(OBSTACLES):
        scene.add_entity(gs.morphs.Box(pos=((x1+x2)/2,(y1+y2)/2,.45),
                                      size = (x2-x1, y2-y1, .9), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.38,.42,.48,1)), name=f'obstacle_{i}')
    for k, (x, y) in STATIONS.items():
        # Device cabinets are deliberately offset from collision-free inspection points.
        scene.add_entity(gs.morphs.Box(pos=(x+.58,y+.50,.36),
                                      size = (.42, .34, .72), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.16,.50,.45,1)), name=f'equipment_{k}')
        scene.add_entity(gs.morphs.Box(pos=(x,y,.012),size=(.22,.22,.024),fixed=True,collision=False),
                         surface=gs.surfaces.Plastic(color=(.98,.76,.12,1)), name=f'inspect_marker_{k}')
    urdf = generate_rover_urdf(output/'pai_rover.urdf')
    rover=scene.add_entity(gs.morphs.URDF(file=str(urdf.resolve()),
                                           pos = (BASE[0], BASE[1], .25),
                                           fixed = False, merge_fixed_links = True),
                           name = 'pai_rover')
    camera = (scene.add_camera(res=settings.res,
                               pos = (15.8, -13.0, 17.8), lookat = (6.0, 4.2, .0), fov = 48, GUI = False)
              if enable_camera else None)
    scene.build()
    indices = [int(rover.get_joint(n).dofs_idx_local[0]) for n in ('left_wheel_joint','right_wheel_joint')]
    rover.set_dofs_kp(np.zeros(2), indices)
    rover.set_dofs_kv(np.array([7., 7.]), indices)
    rover.set_dofs_force_range(lower = np.array([-18., -18.]), upper = np.array([18., 18.]), dofs_idx_local = indices)
    return scene, rover, camera, indices


def run_3d(settings:MissionSettings, detector:Detector, output:Path, smoke = False):
    """3D wheel-actuated closed-loop run. Returns physical metrics, separate from 2D benchmark."""
    scene, rover, camera, joints = make_scene(settings, output)
    scenario = make_scenario(settings.seed)
    p = Patrol(scenario, detector, policy = settings.policy)
    phys_rows = []
    events = []
    visits = []
    path=Path(output)/('genesis_smoke.mp4' if smoke else f'genesis_{settings.policy}.mp4')
    total_s = 2.0 if smoke else settings.duration_s
    steps = round(total_s/settings.dt)
    last_xy = None
    physical_dist = 0.
    boundaries = 0
    ground_contact_flags = 0
    # Ground truth stays in `scenario`; Patrol only uses it in evaluation and onsite checks.
    camera.start_recording(save_to_filename = str(path.resolve()), fps = settings.fps)
    try:
        for i in range(steps):
            p.t = i*settings.dt
            pos, theta, z = xy_yaw(rover)
            p.pos = np.array(pos)
            p.heading = theta
            if last_xy is not None:
                physical_dist += math.dist(pos, last_xy)
            last_xy = pos
            if p.floor.blocked(*pos, margin = .12):
                boundaries += 1 # geofence/proximity proxy, NOT a measured collision impulse
            if z<-.05:
                ground_contact_flags += 1
            if not smoke:
                if p.t + 1e-6 >= p.next_remote:
                    p.observe_remote()
                    p.next_remote += p.cfg.remote_period
                    if settings.policy == 'adaptive' and p.target not in (None,'HOME') and p.t-p.target_start>6:
                        choices = p.options()
                        serious=[k for k in choices if k!=p.target and
                                p.consecutive_alarm[k] >= 2 and
                                p.remote_risks[k]>p.remote_risks.get(p.target, 0)+1.05]
                        if serious:
                            best = max(serious, key = lambda k:p.remote_risks[k]-.18*p.floor.distance(p.pos, STATIONS[k]))
                            p.set_target(best)
                if p.target is None and not p.finished:
                    p.set_target(p.choose_target())
            else:
                if p.target is None:
                    p.set_target('A')
            v = 0.
            omega = 0.
            if p.target is not None and not p.finished:
                target = np.asarray(BASE if p.target == 'HOME' else STATIONS[p.target])
                if np.linalg.norm(target-p.pos)<.40:
                    if smoke:
                        p.finished = True
                    else:
                        p.arrive()
                if not p.finished and p.target is not None:
                    if p.waypoint >= len(p.route):
                        p.route = p.floor.plan(p.pos, target)
                        p.waypoint = 0
                        p.replans += 1
                    wp = np.asarray(p.route[p.waypoint])
                    if np.linalg.norm(wp-p.pos)<.28 and p.waypoint<len(p.route)-1:
                        p.waypoint += 1
                        wp = np.asarray(p.route[p.waypoint])
                    error = (math.atan2(wp[1]-p.pos[1], wp[0]-p.pos[0])-theta+math.pi)%(2*math.pi)-math.pi
                    omega = float(np.clip(2.2*error, -settings.max_omega, settings.max_omega))
                    v = float(settings.max_v*max(0, 1-abs(error)/.80))
            # Twist -> joint velocities. Sign is determined by URDF wheel joint axes.
            wl = settings.wheel_sign*(v-settings.wheel_track*omega/2)/settings.wheel_radius
            wr = settings.wheel_sign*(v+settings.wheel_track*omega/2)/settings.wheel_radius
            rover.control_dofs_velocity(np.asarray([wl, wr]), dofs_idx_local = joints)
            scene.step()
            phys_rows.append(dict(t=p.t,x=pos[0],y=pos[1],z=z,yaw=theta,
                                  v_command = v, omega_command = omega,
                                  target = p.target or 'NONE',
                                  max_ai_risk = max(p.remote_risks.values()),
                                  completed = p.finished))
            if p.finished and not smoke:
                # Continue recording a stationary robot to preserve a 60-180s video.
                rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        # Recording is also stopped in finally.
    finally:
        try:
            rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        except Exception:
            pass
        camera.stop_recording()
    phys = pd.DataFrame(phys_rows)
    phys.to_csv(Path(output)/'genesis_physical_trace.csv', index = False)
    pd.DataFrame(p.visits).to_csv(Path(output)/'genesis_visits.csv', index = False)
    pd.DataFrame(p.history).to_csv(Path(output)/'genesis_ai_risks.csv', index = False)
    # Note: p.events is the policy control event log; 3D distance derived from rigid body position.
    (Path(output)/'genesis_events.json').write_text(json.dumps(p.events, indent = 2, ensure_ascii = False))
    result={'policy':settings.policy,'seed':settings.seed,'simulated_3d_seconds':total_s,
            'physical_distance_m':physical_dist,'inspected_stations':len(p.visits),
            'replans':p.replans,'geofence_proximity_steps_proxy':boundaries,
            'below_floor_steps':ground_contact_flags,'video':str(path),
            'method':'Genesis wheel-joint velocity control and base pose feedback; synthetic telemetry',
            'warning':'Not a real-world robot experiment; geofence is not a contact sensor.'}
    (Path(output)/'genesis_physical_metrics.json').write_text(json.dumps(result, indent = 2, ensure_ascii = False))
    return result, phys, path


Overwriting genesis_mission.py


In [23]:
%%writefile road_ml.py
"""Road issue classification utilities for the PAI final project.

The Hugging Face road issues data has *issue category* labels, not defect
severity or normal-vs-abnormal labels. The curated risk score below is a
policy assumption (not learned severity, not a calibrated probability).
"""
from __future__ import annotations
import json
from pathlib import Path
import numpy as np

REPO_ID = 'Programmer-RD-AI/road-issues-detection-dataset'
RISK_BY_LABEL = {
    'Pothole Issues': 0.95,
    'Damaged Road issues': 0.75,
    'Broken Road Sign Issues': 0.70,
    'Illegal Parking Issues': 0.55,
    'Mixed Issues': 0.85,
    'Littering Garbage on Public Places Issues': 0.40,
    'Vandalism Issues': 0.35,
}


def label_risk(label: str) -> float:
    if label not in RISK_BY_LABEL:
        raise ValueError(f'Unknown dataset label: {label!r}. Revisit the risk policy mapping.')
    return float(RISK_BY_LABEL[label])


def expected_policy_risk(class_probabilities, class_names):
    """Expected *heuristic* risk: not probability of a severe physical hazard."""
    vals = np.asarray([label_risk(str(n)) for n in class_names], dtype = float)
    p = np.asarray(class_probabilities, dtype = float)
    if p.shape[-1] != len(vals):
        raise ValueError('class dimension mismatch')
    return np.sum(p*vals, axis = -1)


def select_station(current_xy, locations, scores, visited = None, policy = 'adaptive'):
    """Distance vs learned-category-dependent policy comparison.

    Note: straight-line distance is for target priority only, not obstacle-safe
    path length. The physical controller independently uses A*.
    """
    visited = set(visited or ())
    available = [s for s in locations if s not in visited]
    if not available:
        return None
    pos = np.asarray(current_xy, dtype = float)
    dd = {k:float(np.linalg.norm(pos-np.asarray(locations[k]))) for k in available}
    if policy == 'nearest':
        return min(available, key = lambda k:(dd[k], k))
    if policy != 'adaptive':
        raise ValueError('policy should be nearest or adaptive')
    return max(available, key = lambda k:(3.5*float(scores[k]) - 0.30*dd[k], -dd[k]))


def safe_speed_limit(risk, nominal_mps = 0.65):
    """Interpretable, deliberately conservative safety shield (simulation only)."""
    risk = float(np.clip(risk, 0., 1.))
    if risk >= 0.80:
        return min(nominal_mps, 0.20)
    if risk >= 0.60:
        return min(nominal_mps, 0.32)
    if risk >= 0.40:
        return min(nominal_mps, 0.48)
    return float(nominal_mps)


def save_station_manifest(items, path):
    Path(path).write_text(json.dumps(items, ensure_ascii = False, indent = 2), encoding = 'utf8')


class RoadVision:
    def __init__(self, checkpoint_path, device = 'cpu'):
        import torch
        import torchvision
        from torchvision.models import mobilenet_v3_small
        ckpt = torch.load(str(checkpoint_path), map_location = 'cpu', weights_only = True)
        self.class_names = [str(x) for x in ckpt['labels']]
        # Model weights from the training notebook; do not silently use ImageNet labels.
        self.model = mobilenet_v3_small(weights = None, num_classes = len(self.class_names))
        self.model.load_state_dict(ckpt['state_dict'])
        self.device = torch.device(device)
        self.model.to(self.device).eval()
        self.preprocess=torchvision.transforms.Compose([
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean = (.485, .456, .406), std = (.229, .224, .225))
        ])
        self.temperature = float(ckpt.get('temperature', 1.0))
        self.source = ckpt.get('source','unknown')

    def predict(self, img):
        import torch
        from PIL import Image
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        x = self.preprocess(img.convert('RGB')).unsqueeze(0).to(self.device)
        with torch.inference_mode():
            probs = torch.softmax(self.model(x)[0]/self.temperature, dim = 0).cpu().numpy()
        predicted = int(np.argmax(probs))
        return {
            'label':self.class_names[predicted],
            'confidence':float(probs[predicted]),
            'policy_risk':float(expected_policy_risk(probs, self.class_names)),
            'category_probabilities':{n:float(p) for n, p in zip(self.class_names, probs)},
            'risk_is_heuristic':True,
        }


Overwriting road_ml.py


## 3. Notebook 04の学習済み資産をアップロードし、AI推論を一度だけ実行
`road_ml_assets.zip` をアップロードします。学習済み重みと画像を読み込み、各設備の予測結果をJSONに保存した後、Genesis側には**予測結果のみ**を渡します。

In [24]:
import zipfile
import gc
import torch
from google.colab import files
from pathlib import Path
asset_dir = Path('ml_assets')
asset_dir.mkdir(exist_ok = True)
if not Path('road_ml_assets.zip').exists():
    received = files.upload()
    del received
assert Path('road_ml_assets.zip').exists(), 'Notebook 04で生成したroad_ml_assets.zipが必要です'
with zipfile.ZipFile('road_ml_assets.zip') as z:
    total = sum(e.file_size for e in z.infolist())
    assert total < 2 * 1024**3, 'Archive too large: inspect contents before extraction'
    for e in z.infolist():
        if e.filename.startswith('/') or '..' in Path(e.filename).parts:
            raise ValueError('Unexpected archive path')
    z.extractall(asset_dir)
from road_ml import RoadVision
from pai_patrol import STATIONS
import json
manifest = json.loads((asset_dir/'station_manifest.json').read_text(encoding = 'utf-8'))
model = RoadVision(asset_dir/'road_mobilenetv3.pt')
feeds = {}
for entry in manifest:
    k = entry['station']
    assert k in STATIONS
    item = model.predict(asset_dir/entry['image'])
    feeds[k]={'label':str(item['label']), 'confidence':float(item['confidence']),
              'policy_risk':float(item['policy_risk']), 'truth_category':str(entry['label']),
              'image_path':str((asset_dir/entry['image']).resolve())}
assert set(feeds) == set(STATIONS)
out = Path('outputs')
out.mkdir(exist_ok = True)
(out/'hf_inference.json').write_text(json.dumps(feeds, ensure_ascii = False, indent = 2), encoding = 'utf-8')
del model
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
for k, r in feeds.items():
    print(k,'pred', r['label'],'risk', round(r['policy_risk'], 3))
print('Inference saved:', out/'hf_inference.json')

A pred Broken Road Sign Issues risk 0.677
B pred Damaged Road issues risk 0.75
C pred Littering Garbage on Public Places Issues risk 0.48
D pred Littering Garbage on Public Places Issues risk 0.407
E pred Mixed Issues risk 0.864
F pred Pothole Issues risk 0.948
Inference saved: outputs/hf_inference.json


## 4. Genesisの安全実行ワーカーを作成
このワーカーはHFキーや学習済みモデル重みを読み込まず、事前に生成したAI分類結果のみを参照します。

In [25]:
%%writefile genesis_safe_worker.py
"""Separate-process Genesis worker; it NEVER handles HF token or PyTorch model weights.
Real Genesis wheel-joint physics; external held-out HF photos provide already-inferred labels.
Two independent episodes are recorded separately, not a continuous mission.
"""
import os
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("PYGLET_HEADLESS", "true")
import argparse
import json
import math
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd

def event(kind, **kw):
    print(json.dumps({"event":kind,"wall_time":time.time(), **kw}, ensure_ascii = False, default = str), flush = True)

def write_json(p, item):
    Path(p).write_text(json.dumps(item, ensure_ascii = False, indent = 2, default = str))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices = ["physics","video"], required = True)
    ap.add_argument("--policy", choices = ["adaptive","nearest"], default = "adaptive")
    ap.add_argument("--duration", type = float, default = 3.)
    ap.add_argument("--fps", type = int, default = 5)
    ap.add_argument("--width", type = int, default = 384)
    ap.add_argument("--height", type = int, default = 288)
    ap.add_argument("--backend", choices = ["cuda","cpu"], default = "cuda")
    ap.add_argument("--output", default = "outputs")
    ap.add_argument("--risk-json", default = "outputs/hf_inference.json")
    ap.add_argument("--label", default = "smoke")
    args = ap.parse_args()
    if args.duration <= 0 or args.duration>60:
        raise ValueError('duration must be 0-60s')
    from genesis_mission import make_scene, xy_yaw, MissionSettings
    from pai_patrol import BASE, STATIONS, FloorMap
    from road_ml import select_station, safe_speed_limit, label_risk
    out = Path(args.output)
    out.mkdir(exist_ok = True, parents = True)
    feed = json.loads(Path(args.risk_json).read_text())
    if set(feed) != set(STATIONS):
        raise ValueError('Missing virtual camera predictions for A..F')
    risks = {k:float(feed[k]['policy_risk']) for k in STATIONS}
    scores = risks if args.policy == 'adaptive' else {k:0. for k in STATIONS}
    event('pre_import_genesis', backend = args.backend, mode = args.mode, seconds = args.duration)
    import torch
    event('torch_state', cuda_available = bool(torch.cuda.is_available()), torch_version = torch.__version__)
    if args.backend == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('GPU is unavailable. Change Colab runtime to GPU and restart before continuing.')
    import genesis as gs
    gs.init(backend = gs.cuda if args.backend == 'cuda' else gs.cpu, logging_level = 'warning', seed = 42)
    event('gs_ready', backend = str(gs.backend), device = str(gs.device))
    cfg = MissionSettings(dt = .05, duration_s = args.duration, fps = args.fps, res = (args.width, args.height), policy = args.policy, seed = 42)
    event('scene_build_begin')
    scene, rover, camera, joints = make_scene(cfg, out, enable_camera = (args.mode == 'video'))
    event('scene_build_done', with_camera = camera is not None)
    floor = FloorMap()
    visited = set()
    target = None
    route = []
    idx = 0
    travel = 0.
    last_xy = None
    first_high_visit = None
    trace = []
    visits = []
    video_path=out/f'{args.label}.mp4'
    if camera is not None:
        event('recording_start', path = str(video_path), fps = args.fps)
        camera.start_recording(save_to_filename = str(video_path.resolve()), fps = args.fps)
    try:
        for step in range(int(round(args.duration/cfg.dt))):
            t = step*cfg.dt
            pos, yaw, z = xy_yaw(rover)
            if last_xy is not None:
                travel += math.dist(pos, last_xy)
            last_xy = pos
            if target is None:
                target = select_station(pos, STATIONS, scores, visited, args.policy)
                if target is not None:
                    route = floor.plan(pos, STATIONS[target])
                    idx = 0
            v = 0.
            omega = 0.
            local_risk = 0.
            if target is not None:
                distance = math.dist(pos, STATIONS[target])
                local_risk = risks[target]*max(0., 1.-distance/3.)
                if distance<.45:
                    visited.add(target)
                    visits.append({'station':target,'arrival_s':t,
                         'groundtruth_category':feed[target]['truth_category'],
                         'predicted_category':feed[target]['label'],'risk_policy':risks[target]})
                    if label_risk(feed[target]['truth_category']) >= .8 and first_high_visit is None:
                        first_high_visit = t
                    target = None
                    route = []
                else:
                    if idx >= len(route):
                        route = floor.plan(pos, STATIONS[target])
                        idx = 0
                    if idx<len(route)-1 and math.dist(pos, route[idx])<.27:
                        idx += 1
                    wp = route[idx]
                    angle = math.atan2(wp[1]-pos[1], wp[0]-pos[0])
                    error = (angle-yaw+math.pi)%(2*math.pi)-math.pi
                    omega = float(np.clip(2.0*error, -cfg.max_omega, cfg.max_omega))
                    # Nearest baseline obeys the same speed cap, isolates prioritization effect.
                    max_v = safe_speed_limit(local_risk, cfg.max_v)
                    v = max_v*max(0., 1.-abs(error)/.8)
            wl = cfg.wheel_sign*(v-.5*cfg.wheel_track*omega)/cfg.wheel_radius
            wr = cfg.wheel_sign*(v+.5*cfg.wheel_track*omega)/cfg.wheel_radius
            rover.control_dofs_velocity(np.asarray([wl, wr]), dofs_idx_local = joints)
            scene.step()
            trace.append({'time_s':t,'x':pos[0],'y':pos[1],'z':z,'target':target or 'NONE',
                          'risk_from_hf':local_risk,'speed_cmd_mps':v,'visited':len(visited)})
            if (step+1)%20 == 0:
                row={'completed_sim_s':(step+1)*cfg.dt,'policy':args.policy,'mode':args.mode,
                     'last_target':target,'physical_distance_m':travel,'wall_time':time.time()}
                write_json(out/f'{args.label}_heartbeat.json',row)
                # For hard crashes, preserve partial trace and print flushable progress.
                pd.DataFrame(trace).to_csv(out/f'{args.label}_trace_partial.csv',index=False)
                event('progress', step = step+1, sim_s = row['completed_sim_s'], meters = round(travel, 2))
    finally:
        try:
            rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        except Exception:
            pass
        if camera is not None:
            event('recording_stop_begin')
            camera.stop_recording()
            event('recording_stop_done')
    metrics={'policy':args.policy,'mode':args.mode,'physical_distance_m':travel,
             'visited':len(visited),'first_high_risk_visit_s':first_high_visit,
             'duration_sim_s':args.duration,'video':str(video_path) if camera is not None else None,
             'external_virtual_camera_images':True,'genesis_onboard_vision':False}
    pd.DataFrame(trace).to_csv(out/f'{args.label}_physical_trace.csv',index=False)
    pd.DataFrame(visits).to_csv(out/f'{args.label}_visits.csv',index=False)
    write_json(out/f'{args.label}_metrics.json',metrics)
    event('worker_success', metrics = metrics)
    return 0

if __name__ == '__main__':
    try:
        sys.exit(main())
    except Exception as exc:
        event('worker_exception',error=f'{type(exc).__name__}: {exc}')
        raise

Overwriting genesis_safe_worker.py


## 5. カーネル保護付き実行関数
クラッシュの直前に `outputs/*.log` と `*_heartbeat.json` が残り、失敗した段階を絞り込めます。`-9` や `-11` はSIGKILL/SIGSEGV等の可能性がありますが、それだけでOOMを断定しません。

In [26]:
from pathlib import Path
import time
import subprocess
import json
import os
import psutil
import sys
out = Path('outputs')
out.mkdir(exist_ok = True)

def tail(path, lines = 28):
    if not Path(path).exists():
        return ''
    return '\n'.join(Path(path).read_text(errors = 'replace').splitlines()[-lines:])

def run_stage(label, *, mode = 'physics', policy = 'adaptive', duration = 2.0, fps = 5, width = 384, height = 288, timeout = 900):
    # GPU backend is mandatory for actual Colab mission; no quiet CPU fallback.
    args=[sys.executable,'-u','genesis_safe_worker.py','--mode',mode,
          '--policy', policy,'--duration', str(duration),'--fps', str(fps),
          '--width', str(width),'--height', str(height),'--backend','cuda',
          '--risk-json', str(out/'hf_inference.json'),'--output', str(out),'--label', label]
    logfile=out/f'{label}.log'
    print('START', label, mode, policy, duration,'s', width,'x', height,'at', fps,'fps')
    t0 = time.monotonic()
    status = None
    env = os.environ.copy()
    env['PYOPENGL_PLATFORM'] = os.environ.get('PAI_GL','egl')
    env.setdefault('PYGLET_HEADLESS','true')
    with logfile.open('w') as fh:
        try:
            process=subprocess.run(args,stdout=fh,stderr=subprocess.STDOUT,
                                   env = env, timeout = timeout, check = False)
            status = process.returncode
        except subprocess.TimeoutExpired:
            status = 'timeout'
    print('END', label,'returncode:', status,'walltime_s:', round(time.monotonic()-t0, 1))
    print(tail(logfile, 35))
    if status != 0:
        hb=out/f'{label}_heartbeat.json'
        if hb.exists():
            print('Partial heartbeat:', hb.read_text()[:500])
        raise RuntimeError(f'{label} failed ({status}). Download outputs/{label}.log and do not run subsequent stages.')
    if mode == 'video':
        video=out/f'{label}.mp4'
        if not video.exists() or video.stat().st_size<1000:
            raise RuntimeError('MP4 was not created: '+str(video))
    return logfile

## 6. Stage A：2秒・物理計算のみ（動画なし）
ここで落ちたらURDF/Scene build/物理演算の問題が主候補です。**カメラを作らなくてもGenesis内部でOpenGLコンテキストが初期化される場合があり、描画要因を完全には排除できません。**

In [27]:
run_stage('A_physics_2s', mode = 'physics', duration = 2., timeout = 600)

START A_physics_2s physics adaptive 2.0 s 384 x 288 at 5 fps
END A_physics_2s returncode: 0 walltime_s: 58.0
{"event": "pre_import_genesis", "wall_time": 1792130113.2761497, "backend": "cuda", "mode": "physics", "seconds": 2.0}
{"event": "torch_state", "wall_time": 1792130113.922428, "cuda_available": true, "torch_version": "2.11.0+cu130"}
{"event": "gs_ready", "wall_time": 1792130118.2730987, "backend": "2", "device": "cuda:0"}
{"event": "scene_build_begin", "wall_time": 1792130118.2731576}
/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node = ast.keyword(
/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant i

PosixPath('outputs/A_physics_2s.log')

## 7. Stage B：3秒・低負荷レンダリング
Stage A成功後に実行。こちらだけで落ちたらEGL・OpenGL・カメラレンダリングの問題が考えられます。

In [28]:
run_stage('B_render_3s', mode = 'video', duration = 3., fps = 5, width = 320, height = 240, timeout = 900)
from IPython.display import Video, display
display(Video('outputs/B_render_3s.mp4', embed = False))

START B_render_3s video adaptive 3.0 s 320 x 240 at 5 fps
END B_render_3s returncode: 0 walltime_s: 12.6
{"event": "pre_import_genesis", "wall_time": 1792130171.325864, "backend": "cuda", "mode": "video", "seconds": 3.0}
{"event": "torch_state", "wall_time": 1792130171.970494, "cuda_available": true, "torch_version": "2.11.0+cu130"}
{"event": "gs_ready", "wall_time": 1792130176.2904668, "backend": "2", "device": "cuda:0"}
{"event": "scene_build_begin", "wall_time": 1792130176.2905219}
{"event": "scene_build_done", "wall_time": 1792130180.3393674, "with_camera": true}
{"event": "recording_start", "wall_time": 1792130180.340354, "path": "outputs/B_render_3s.mp4", "fps": 5}
{"event": "progress", "wall_time": 1792130181.1341941, "step": 20, "sim_s": 1.0, "meters": 0.1}
{"event": "progress", "wall_time": 1792130181.1617703, "step": 40, "sim_s": 2.0, "meters": 0.19}
{"event": "progress", "wall_time": 1792130181.1896853, "step": 60, "sim_s": 3.0, "meters": 0.27}
{"event": "recording_stop_begi

## 8. Stage C：30秒×2本の独立比較実験
**先にStage A/Bが成功していること。** それぞれ別プロセスで構築・終了します。32秒を記録して先頭30秒を提出用に使用し、端数不足を避けます。640×480・20fps → 384×288・5fpsに負荷を低減しています。実験は `nearest`（最短近傍を優先）と `adaptive`（学習済み分類に基づくリスク優先）です。

In [29]:
# 1本目: AIなし近傍優先（画像リスクは走行速度制限のみ共通）
run_stage('C_nearest_30s', mode = 'video', policy = 'nearest', duration = 32., fps = 5, width = 384, height = 288, timeout = 1500)

START C_nearest_30s video nearest 32.0 s 384 x 288 at 5 fps
END C_nearest_30s returncode: 0 walltime_s: 11.5
{"event": "progress", "wall_time": 1792130192.0693305, "step": 20, "sim_s": 1.0, "meters": 0.1}
{"event": "progress", "wall_time": 1792130192.0966733, "step": 40, "sim_s": 2.0, "meters": 0.19}
{"event": "progress", "wall_time": 1792130192.1244066, "step": 60, "sim_s": 3.0, "meters": 0.27}
{"event": "progress", "wall_time": 1792130192.1522944, "step": 80, "sim_s": 4.0, "meters": 0.33}
{"event": "progress", "wall_time": 1792130192.1805124, "step": 100, "sim_s": 5.0, "meters": 0.4}
{"event": "progress", "wall_time": 1792130192.208714, "step": 120, "sim_s": 6.0, "meters": 0.48}
{"event": "progress", "wall_time": 1792130192.2377493, "step": 140, "sim_s": 7.0, "meters": 0.56}
{"event": "progress", "wall_time": 1792130192.2666302, "step": 160, "sim_s": 8.0, "meters": 0.62}
{"event": "progress", "wall_time": 1792130192.2949462, "step": 180, "sim_s": 9.0, "meters": 0.69}
{"event": "progr

PosixPath('outputs/C_nearest_30s.log')

In [30]:
# 2本目: HF学習済み分類モデルのリスクで優先順位を変更
run_stage('D_adaptive_30s', mode = 'video', policy = 'adaptive', duration = 32., fps = 5, width = 384, height = 288, timeout = 1500)

START D_adaptive_30s video adaptive 32.0 s 384 x 288 at 5 fps
END D_adaptive_30s returncode: 0 walltime_s: 11.5
{"event": "progress", "wall_time": 1792130203.5586977, "step": 20, "sim_s": 1.0, "meters": 0.1}
{"event": "progress", "wall_time": 1792130203.586362, "step": 40, "sim_s": 2.0, "meters": 0.19}
{"event": "progress", "wall_time": 1792130203.6143951, "step": 60, "sim_s": 3.0, "meters": 0.27}
{"event": "progress", "wall_time": 1792130203.6424549, "step": 80, "sim_s": 4.0, "meters": 0.33}
{"event": "progress", "wall_time": 1792130203.6707435, "step": 100, "sim_s": 5.0, "meters": 0.4}
{"event": "progress", "wall_time": 1792130203.6992645, "step": 120, "sim_s": 6.0, "meters": 0.48}
{"event": "progress", "wall_time": 1792130203.7288005, "step": 140, "sim_s": 7.0, "meters": 0.56}
{"event": "progress", "wall_time": 1792130203.7584074, "step": 160, "sim_s": 8.0, "meters": 0.62}
{"event": "progress", "wall_time": 1792130203.787126, "step": 180, "sim_s": 9.0, "meters": 0.69}
{"event": "pro

PosixPath('outputs/D_adaptive_30s.log')

## 9. 2実験を明示した60秒比較動画を生成
ここでは**Genesisシミュレーションを再実行しません**。既存のMP4に字幕を追加するだけです。提出用1分動画は「Nearest 30秒 → AI adaptive 30秒」であり、連続した1回の走行と誤解させません。

In [31]:
import cv2
import json
import pandas as pd
from pathlib import Path
from IPython.display import Video, display
out = Path('outputs')
output_file = out/'PAI_final_task_HF_Genesis_crash_safe.mp4'
video_files=[(out/'C_nearest_30s.mp4','BASELINE: NEAREST TARGET'),
             (out/'D_adaptive_30s.mp4','HF MODEL: RISK ADAPTIVE')]
size = (384, 288)
fps_expected = 5.0
w = cv2.VideoWriter(str(output_file), cv2.VideoWriter_fourcc(*'mp4v'), fps_expected, size)
assert w.isOpened(),'MP4 writer could not be initialized'
all_frames = 0
try:
    for idx, (file, label) in enumerate(video_files):
        cap = cv2.VideoCapture(str(file))
        assert cap.isOpened(), str(file)
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(label,'input fps', fps,'frames', n)
        assert abs(fps-fps_expected)<.01 and n>=150, f'Expected at least 30s at 5fps, got {n/fps:.2f}s'
        tr=pd.read_csv(out/f'{file.stem}_physical_trace.csv')
        cur = 0
        while True:
            ok, frame = cap.read()
            if not ok or cur >= 150:
                break
            frame = cv2.resize(frame, size)
            row = tr.iloc[min(len(tr)-1, int(cur/5/.05))]
            cv2.rectangle(frame, (0, 0), (384, 78), (18, 24, 32), -1)
            cv2.putText(frame, label, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, .52, (255, 255, 255), 1)
            cv2.putText(frame,f'EPISODE {idx+1}/2 | SIM {cur/5:.1f}s | TARGET {row["target"]}',(8,41),cv2.FONT_HERSHEY_SIMPLEX,.37,(205,230,253),1)
            cv2.putText(frame,f'AI: external HF photo classification | risk {float(row["risk_from_hf"]):.2f}',(8,61),cv2.FONT_HERSHEY_SIMPLEX,.34,(190,220,180),1)
            w.write(frame)
            cur += 1
            all_frames += 1
        cap.release()
        print('appended', cur,'frames')
finally:
    w.release()
actual = all_frames/fps_expected
print('FINAL:', output_file,'duration', actual,'seconds')
assert 60 <= actual <= 180, '1〜3分の提出規定を満たしません。録画のfps・尺を確認してください'
display(Video(str(output_file), embed = False))

BASELINE: NEAREST TARGET input fps 5.0 frames 160
appended 150 frames
HF MODEL: RISK ADAPTIVE input fps 5.0 frames 160
appended 150 frames
FINAL: outputs/PAI_final_task_HF_Genesis_crash_safe.mp4 duration 60.0 seconds


## 10. トレース・指標・診断ログをZIPで保存
`outputs` フォルダにはモデルの予測結果、実験CSV・JSON、短い診断ログ、3D実験MP4が入ります。**クラッシュ時はログだけでも先にダウンロード**してください。

In [32]:
import shutil
from pathlib import Path
archive = shutil.make_archive('Genesis_06_outputs_crash_safe','zip', root_dir = 'outputs')
print('Diagnostics:', archive)
from google.colab import files
files.download(archive)
# 提出するのは最終確認した MP4 のみ: files.download('outputs/PAI_final_task_HF_Genesis_crash_safe.mp4')

Diagnostics: /content/Genesis_06_outputs_crash_safe.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### クラッシュした場合の共有情報
`outputs/A_physics_2s.log` / `B_render_3s.log` / `C_nearest_30s.log` / `D_adaptive_30s.log` のうち失敗したものを共有してください。`nvidia-smi`出力もあると判別しやすいです。**`HFH_READ` の値は不要**です。